# Flow Tape Analysis — SPXW 0DTE (2026-06-11)

Step-by-step calculation of the Flow Tape indicator from raw SPXW option chain snapshots.

**Method:**
1. Load all 1-min snapshots for the 0DTE expiry on the sample date
2. Filter to rows where `last` is inside `[bid, ask]` (drop stale prints)
3. Apply contract filter: all OTM strikes + 25 nearest ITM strikes per side
4. Per contract: compute `trade_position`, EMA-smooth it, derive `trade_direction`
5. Compute `new_volume` (lookback-window volume delta)
6. `flow = new_volume × trade_direction × |delta|`
7. Aggregate per timestamp by contract type → cumulative call/put flow lines

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

from datetime import UTC, date, datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from trade_dash.config import OPTIONS_DIR
from trade_dash.data.options import find_snapshots_for_expiry_on_date, load_options_snapshot

CHICAGO = ZoneInfo('America/Chicago')

def to_chicago(ts: datetime) -> datetime:
    if ts.tzinfo is None:
        ts = ts.replace(tzinfo=UTC)
    return ts.astimezone(CHICAGO).replace(tzinfo=None)

## Parameters

In [ ]:
SAMPLE_DATE   = date(2026, 6, 11)
EXPIRY        = date(2026, 6, 11)   # 0DTE
SYMBOL        = 'SPXW'
LOOKBACK      = 5    # minutes (snapshot intervals) for new-volume delta
EMA_SPAN      = 20   # EMA lookback for trade_position smoothing
ITM_LIMIT     = 25   # max ITM strikes to include per contract side

## 1. Load snapshots

In [ ]:
snapshots = find_snapshots_for_expiry_on_date(
    SYMBOL, expiry=EXPIRY, sample_date=SAMPLE_DATE, data_dir=OPTIONS_DIR
)
print(f'Snapshots loaded: {len(snapshots)}')
print(f'First: {to_chicago(snapshots[0][0]).strftime("%H:%M:%S CT")}  '
      f'Last: {to_chicago(snapshots[-1][0]).strftime("%H:%M:%S CT")}')

## 2. Concatenate all snapshots into one DataFrame

In [ ]:
frames = []
for ts, path in snapshots:
    df = load_options_snapshot(path).copy()
    df['_ts_utc'] = ts
    df['_ts_ct']  = to_chicago(ts)
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
print(f'Total rows (all snapshots): {len(raw):,}')
print(f'Columns: {list(raw.columns)}')

## 3. Clean: coerce numerics, uppercase contract_type, drop bad rows

In [ ]:
for col in ['bid', 'ask', 'last', 'total_volume', 'delta', 'strike']:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')

raw['contract_type'] = raw['contract_type'].str.upper()
raw = raw.dropna(subset=['bid', 'ask', 'last', 'total_volume', 'delta', 'strike'])
raw = raw[raw['total_volume'] > 0]

print(f'Rows after dropna + volume>0: {len(raw):,}')

## 4. Drop stale prints: require `last` inside `[bid, ask]`

When a strike hasn't traded recently, the exchange feed keeps reporting the last
trade price from minutes (or hours) ago. As the quote moves away from that stale
print, `last` ends up outside the current spread, making `trade_position`
meaningless. Filtering these rows was the key fix for the sign-inversion bug.

In [ ]:
n_before = len(raw)
last_in_spread = (raw['last'] >= raw['bid']) & (raw['last'] <= raw['ask'])
raw = raw[last_in_spread].copy()

pct_dropped = (n_before - len(raw)) / n_before
print(f'Rows with stale last (outside spread): {n_before - len(raw):,}  ({pct_dropped:.1%} of rows)')
print(f'Rows remaining: {len(raw):,}')

## 5. Determine spot price and apply strike filter

Keep: all OTM strikes + the `ITM_LIMIT` nearest ITM strikes per contract side.
This matches the Flow Tape spec and avoids deep-ITM contracts with tiny spreads
and high deltas that would dominate the signal.

In [ ]:
# Spot from the last snapshot
spot = float(
    pd.to_numeric(load_options_snapshot(snapshots[-1][1])['underlying_price'], errors='coerce')
    .dropna().iloc[0]
)
print(f'Spot (from last snapshot): {spot:.2f}')

call_mask = raw['contract_type'] == 'CALL'
put_mask  = raw['contract_type'] == 'PUT'

call_strikes = raw.loc[call_mask, 'strike']
put_strikes  = raw.loc[put_mask,  'strike']

# Calls: OTM = strike > spot; ITM = strike <= spot, keep ITM_LIMIT nearest
call_otm = set(call_strikes[call_strikes > spot].unique())
call_itm = set(sorted(call_strikes[call_strikes <= spot].unique(), reverse=True)[:ITM_LIMIT])
allowed_calls = call_otm | call_itm

# Puts: OTM = strike < spot; ITM = strike >= spot, keep ITM_LIMIT nearest
put_otm = set(put_strikes[put_strikes < spot].unique())
put_itm = set(sorted(put_strikes[put_strikes >= spot].unique())[:ITM_LIMIT])
allowed_puts = put_otm | put_itm

strike_mask = (
    (call_mask & raw['strike'].isin(allowed_calls)) |
    (put_mask  & raw['strike'].isin(allowed_puts))
)
filtered = raw[strike_mask].copy()

print(f'Allowed call strikes: {len(allowed_calls)}  put strikes: {len(allowed_puts)}')
print(f'Rows after strike filter: {len(filtered):,}')

## 6. Per-contract flow calculation

For each `(strike, expiration_date, contract_type)` group:

```
trade_position  = (last - bid) / (ask - bid).clip(0.01)   → [0, 1]
ema_tp          = EMA_SPAN(trade_position)
trade_direction = (ema_tp - 0.5) × 2                       → [-1, +1]
new_volume      = total_volume - total_volume.shift(LOOKBACK)
flow            = new_volume × trade_direction × |delta|
```

In [ ]:
contract_cols = ['strike', 'expiration_date', 'contract_type']
filtered = filtered.sort_values(contract_cols + ['_ts_utc'])

flow_rows = []

for _, grp in filtered.groupby(contract_cols, sort=False):
    grp = grp.sort_values('_ts_utc').copy()

    spread          = (grp['ask'] - grp['bid']).clip(lower=0.01)
    trade_position  = ((grp['last'] - grp['bid']) / spread).clip(0.0, 1.0)
    ema_tp          = trade_position.ewm(span=EMA_SPAN, adjust=False).mean()
    trade_direction = (ema_tp - 0.5) * 2

    new_volume = (
        grp['total_volume'] - grp['total_volume'].shift(LOOKBACK)
    ).clip(lower=0.0).fillna(0.0)

    flow = new_volume * trade_direction * grp['delta'].abs()

    for ts_ct, ts_utc, fv, ct in zip(
        grp['_ts_ct'], grp['_ts_utc'], flow, grp['contract_type'], strict=True
    ):
        if pd.isna(fv):
            continue
        flow_rows.append({
            '_ts_ct':        ts_ct,
            '_ts_utc':       ts_utc,
            'contract_type': ct,
            'flow':          float(fv),
        })

flow_df = pd.DataFrame(flow_rows)
print(f'Flow rows computed: {len(flow_df):,}')
flow_df.head(10)

## 7. Aggregate per timestamp and cumsum

In [ ]:
agg = (
    flow_df
    .groupby(['_ts_ct', 'contract_type'])['flow']
    .sum()
    .reset_index()
)

all_ts    = sorted(agg['_ts_ct'].unique())
ts_index  = pd.Index(all_ts, name='_ts_ct')

call_agg = (
    agg[agg['contract_type'] == 'CALL']
    .set_index('_ts_ct')['flow']
    .reindex(ts_index, fill_value=0.0)
)
put_agg = (
    agg[agg['contract_type'] == 'PUT']
    .set_index('_ts_ct')['flow']
    .reindex(ts_index, fill_value=0.0)
)

tape = pd.DataFrame({
    'call_flow_raw':  call_agg,
    'put_flow_raw':   put_agg,
    'call_cumflow':   call_agg.cumsum(),
    'put_cumflow':    put_agg.cumsum(),
})

print(f'Timestamps: {len(tape)}')
print(f'Call cumflow range: [{tape["call_cumflow"].min():,.0f}, {tape["call_cumflow"].max():,.0f}]')
print(f'Put  cumflow range: [{tape["put_cumflow"].min():,.0f},  {tape["put_cumflow"].max():,.0f}]')
tape.tail(10)

## 8. Plot: Flow Tape

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
fig.patch.set_facecolor('#0e0e0e')
for ax in axes:
    ax.set_facecolor('#0e0e0e')
    ax.tick_params(colors='#cccccc')
    ax.xaxis.label.set_color('#cccccc')
    ax.yaxis.label.set_color('#cccccc')
    ax.title.set_color('#eeeeee')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')
    ax.grid(True, color='#222222', linewidth=0.5, linestyle='--')

# --- Top panel: cumulative flow ---
ax = axes[0]
ax.plot(tape.index, tape['call_cumflow'], color='#00c850', linewidth=1.5, label='Call Flow')
ax.plot(tape.index, tape['put_cumflow'],  color='#dc3c3c', linewidth=1.5, label='Put Flow')
ax.axhline(0, color='#555555', linewidth=0.8, linestyle=':')
ax.set_title(
    f'Flow Tape — {SYMBOL} {EXPIRY} (0DTE)  |  '
    f'lookback={LOOKBACK}min  ema_span={EMA_SPAN}',
    fontsize=12, pad=10
)
ax.set_ylabel('Cumulative Flow', fontsize=10)
ax.legend(facecolor='#1a1a1a', edgecolor='#444444', labelcolor='#cccccc', fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# --- Bottom panel: raw per-snapshot flow bars ---
ax2 = axes[1]
ax2.bar(tape.index, tape['call_flow_raw'].clip(lower=0),
        width=0.0006, color='#00c850', alpha=0.7, label='Call (buy)')
ax2.bar(tape.index, tape['call_flow_raw'].clip(upper=0),
        width=0.0006, color='#00641e', alpha=0.7, label='Call (sell)')
ax2.bar(tape.index, tape['put_flow_raw'].clip(lower=0),
        width=0.0006, color='#dc3c3c', alpha=0.7, label='Put (buy)')
ax2.bar(tape.index, tape['put_flow_raw'].clip(upper=0),
        width=0.0006, color='#640000', alpha=0.7, label='Put (sell)')
ax2.axhline(0, color='#555555', linewidth=0.8, linestyle=':')
ax2.set_ylabel('Raw Flow / bar', fontsize=9)
ax2.set_xlabel('Time (CT)', fontsize=10)
ax2.legend(facecolor='#1a1a1a', edgecolor='#444444', labelcolor='#cccccc',
           fontsize=8, ncol=4, loc='upper left')

# X-axis formatting
fmt = mdates.DateFormatter('%H:%M')
axes[1].xaxis.set_major_formatter(fmt)
axes[1].xaxis.set_major_locator(mdates.HourLocator(interval=1))
axes[1].xaxis.set_minor_locator(mdates.MinuteLocator(byminute=[0, 15, 30, 45]))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=0, ha='center')

plt.tight_layout()
plt.savefig('../notebooks/flow_tape_2026-06-11.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved to notebooks/flow_tape_2026-06-11.png')

## 9. Sanity check: per-snapshot call flow around the 12:30 catalyst

In [ ]:
window = tape.between_time('12:00', '13:30')
print('Time              Call raw    Call cum    Put raw     Put cum')
print('-' * 65)
for ts, row in window.iterrows():
    print(
        f'{ts.strftime("%H:%M")}   '
        f'{row["call_flow_raw"]:>10,.0f}  '
        f'{row["call_cumflow"]:>10,.0f}  '
        f'{row["put_flow_raw"]:>10,.0f}  '
        f'{row["put_cumflow"]:>10,.0f}'
    )